# Cross-context generalization — the findings are not NSCLC artifacts

**NOT FOR CLINICAL USE.** Population/trial-level only; illustrative, unverified parameters. Executed in CI (nbmake).

By v0.28 the project's headline results — the resistance-mechanism divergence axis, the model-selection budget's survival-link axis, the conditional ORR→OS surrogate, and depth≠durability — all lived in one context: NSCLC first line. v0.29 gives breast, CRC, HCC, and melanoma the same two pieces NSCLC had (a mechanistic two-population model + a tail-sensitive k_g OS link) and shows every finding reproduces across all five solid-tumor contexts. See `docs/specs/research/cross-context-generalization.md`.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import onkos
from onkos.budget import model_selection_budget, eligible_survival_links

ds = onkos.load()
t = np.linspace(0, 312, 625)
CONTEXTS = ['NSCLC', 'breast', 'CRC', 'HCC', 'melanoma']
print(f'{len(ds)} records | onkos {onkos.__version__}')

## Each first-line context now has a 3-model / 2-link layout

A mechanistic two-population resistance model (universal response, fast resistant regrowth → broad but brief) and a non-default k_g OS link (the tail-sensitive endpoint) were added to each of the four non-NSCLC contexts.

In [ ]:
for tt in CONTEXTS:
    ctx = {'tumor_type': tt, 'line': 'first'}
    cmp = onkos.compare(ds, purpose='tgi', context=ctx, t=t)
    links = eligible_survival_links(ds, ctx, 'OS')
    print(f'{tt:9} {len(cmp.included)} TGI models, {len(links)} OS links')

## The ORR → OS surrogate inverts under k_g — in every context

ORR and the week-8 surrogate are both shrinkage-based, so ORR ranks OS faithfully under week-8. Under the tail-sensitive k_g link, the two-population model (highest ORR, shortest DoR) has the worst OS — ORR inverts. This now reproduces in all five contexts.

In [ ]:
from onkos.response import response_vs_survival
print(f"{'context':<10}{'wk8 disc':>10}{'k_g disc':>10}   ORR predicts OS?")
for tt in CONTEXTS:
    ctx = {'tumor_type': tt, 'line': 'first'}
    kg = f'survival_link.{tt.lower()}_os_growth_rate'
    w = response_vs_survival(ds, context=ctx, t=t, n=200)
    k = response_vs_survival(ds, context=ctx, survival_link=kg, t=t, n=200)
    print(f'{tt:<10}{w.discordant_fraction:>10.2f}{k.discordant_fraction:>10.2f}   '
          f'week-8: {w.orr_predicts_os}, k_g: {k.orr_predicts_os}')
    assert w.orr_predicts_os and not k.orr_predicts_os

## Depth ≠ durability, and the budget survival-link axis — everywhere

In every context the highest-ORR model has the shortest DoR, and the budget's survival-link axis (empty before v0.29) is now a real, often dominant, share of the forecast variance.

In [ ]:
print(f"{'context':<10}{'top ORR':>8}{'its DoR':>8}{'V_link':>8}{'structural':>12}  dominant")
for tt in CONTEXTS:
    ctx = {'tumor_type': tt, 'line': 'first'}
    kg = f'survival_link.{tt.lower()}_os_growth_rate'
    rs = response_vs_survival(ds, context=ctx, survival_link=kg, t=t, n=200)
    rows = [r for r in rs.rows if r['median_dor_weeks'] is not None]
    top = max(rows, key=lambda r: r['orr'])
    b = model_selection_budget(ds, context=ctx, endpoint='OS', n=120)
    print(f"{tt:<10}{top['orr']:>8.2f}{top['median_dor_weeks']:>8.0f}"
          f"{b.fractions['survival_link']*100:>7.0f}%{b.structural_fraction*100:>11.0f}%  {b.dominant}")

## The takeaway

Four single-context demonstrations are now a general claim: across NSCLC, breast, CRC, HCC, and melanoma, the resistance-*mechanism* choice, the survival-*metric* choice, the ORR→OS surrogate's conditionality, and depth-vs-durability all behave the same way. CI enforces it (`tests/test_response.py`, `tests/test_budget.py`).

In [ ]:
# The model-selection budget report now flags 5/6 contexts as structure-dominated.
from onkos.report import model_selection_budget_summary
for row in model_selection_budget_summary(ds):
    flag = '' if row['n_links'] > 1 else '  (single link)'
    print(f"{row['tumor_type']:9} {row['line']:7} {row['n_models']}x{row['n_links']}  {row['dominated_by']}-dominated{flag}")